In [ ]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

# Fase 2B: Limpieza de datos
La limpieza de los datos se realizará considerando los análisis y conclusiones de la fase anterior (para más detalles, consultar el notebook `Fase_1_2A_setup_y_EDA.ipynb`)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, chi2_contingency
from src.carga import cargar_csv
from src.eda_utils import *

In [ ]:
df = cargar_csv("..\data\\raw\Student_Depression_Dataset_Original.csv")

In [ ]:
df_copia = df.copy()

### Eliminando columnas numéricas irrelevantes

In [ ]:
df_copia.drop(columns=["Job Satisfaction", "Work Pressure"], inplace=True)

### Eliminando outliers

In [ ]:
columnas_numericas = df_copia.select_dtypes(include='number').columns

for col in columnas_numericas:

    mascara, li, ls = eliminar_outliers_3sigmas(df_copia[col])

    print(f"Procesando columna: {col}")
    print(f"Outliers detectados: {mascara.sum()}")

    df_copia = df_copia[~mascara]

    print("-"*40)

### Inputación en `Financial Stress`

In [ ]:
df_copia["Financial Stress"].fillna(df_copia["Financial Stress"].median(), inplace=True)

### Eliminación de datos irrelevantes en categóricos 

In [ ]:
#Se eliminan las ciudades que tengan menos de 5 registros
counts = df_copia["City"].value_counts()
ciudades_validas = counts[counts >= 5].index

df_copia = df_copia[df_copia['City'].isin(ciudades_validas)]

In [ ]:
#Se eliminan los "Others" en Sleep Duration y Dietary Habits
df_copia = df_copia[df_copia["Dietary Habits"] != "Others"]
df_copia = df_copia[df_copia["Sleep Duration"] != "Others"]

In [ ]:
#Eliminamos Profession, puesto que no será relevante
df_copia.drop(columns=["Profession"], inplace=True)

### Análisis post limpieza
¿Hubo una pérdida muy grande de datos?

In [ ]:
print("Filas originales:", df.shape[0])
print("Filas después de limpieza:", df_copia.shape[0])
print("Diferencia:",  df.shape[0] - df_copia.shape[0])
print("--"*40)
print("Columnas originales:", df.shape[1])
print("Filas después de eliminar outliers:", df_copia.shape[1])
print("Diferencia:",  df.shape[1] - df_copia.shape[1])

In [ ]:
#¿Cuánto representa en porcentaje?
print("Se eliminó un", round(84 * 100 / 27901, 2) ,"% de datos") 

Comprobamos haber logrado un dataset más limpio, con la eliminación de únicamente el $0,3\%$ de los datos originales.

### Exportación

In [ ]:
df_copia.to_csv("..\data\processed\Student_Depression_Dataset_Limpio.csv", index=False)